In [1]:
import umap.umap_ as umap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import random
from sklearn.discriminant_analysis import StandardScaler
import os
from sklearn.linear_model import LinearRegression
from sklearn.cluster import DBSCAN, AgglomerativeClustering
import statsmodels.api as sm

In [2]:
data = pd.read_csv('edited_data.csv')
data[data < 0] = 0
data

,Unnamed: 0,Date,Direction,Time,TRAFFIC,PRCP,SNOW,SNWD,SNOW_DAY_SUM,Vehicles,Driver Age,Condition_Code,MorF,DayOfWeek
0,0,20140101,1,0,102,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
1,1,20140101,0,0,131,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
2,2,20140101,1,1,91,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
3,3,20140101,0,1,162,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
4,4,20140101,1,2,86,0.12,1.9,13.0,1.9,1.0,26.0,0.0,0.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167291,167291,20241231,0,21,316,0.00,0.0,15.0,8.5,0.0,0.0,0.0,0.0,2
167292,167292,20241231,1,22,281,0.00,0.0,15.0,8.5,1.0,19.0,6.0,0.0,2
167293,167293,20241231,0,22,233,0.00,0.0,15.0,8.5,1.0,19.0,6.0,0.0,2
167294,167294,20241231,1,23,201,0.00,0.0,15.0,8.5,0.0,0.0,0.0,0.0,2


In [3]:
crash_data = data[data['Vehicles'] > 0]
norm_data = data[data['Vehicles'] == 0]

In [4]:
crash_data_mean = crash_data['TRAFFIC'].mean()
crash_data_mean

1370.3840977326845

In [5]:
norm_data_mean = norm_data['TRAFFIC'].mean()
norm_data_mean

1046.851583067427

In [6]:
crash_data_describe = crash_data.describe()

In [7]:
norm_data_describe = norm_data.describe()

In [8]:
crash_data_describe_transpose = crash_data_describe.transpose()
norm_data_describe_transpose = norm_data_describe.transpose()

In [9]:
norm_data_describe_transpose

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,157637.0,8.363740e+04,48244.349358,0.0,41927.0,83623.0,125364.00,167295.00
Date,157637.0,2.019104e+07,32589.473215,20140101.0,20160917.0,20190227.0,20220731.00,20241231.00
Direction,157637.0,4.997748e-01,0.500002,0.0,0.0,0.0,1.00,1.00
Time,157637.0,1.142554e+01,6.974226,0.0,5.0,11.0,18.00,23.00
TRAFFIC,157637.0,1.046852e+03,895.987076,0.0,241.0,859.0,1601.00,9317.00
PRCP,157637.0,3.532540e-02,0.101180,0.0,0.0,0.0,0.01,1.21
SNOW,157637.0,3.824013e-01,1.201151,0.0,0.0,0.0,0.00,18.00
SNWD,157637.0,3.106654e+00,8.002410,0.0,0.0,0.0,0.00,45.50
SNOW_DAY_SUM,157637.0,2.793287e+00,4.631855,0.0,0.0,0.5,3.80,40.10
Vehicles,157637.0,0.000000e+00,0.000000,0.0,0.0,0.0,0.00,0.00


In [10]:
crash_data_describe_count = crash_data_describe_transpose['mean']
norm_data_describe_count = norm_data_describe_transpose['mean']

In [11]:
compared_data = (crash_data_describe_count - norm_data_describe_count)

In [12]:
compared_data = pd.DataFrame(compared_data)

In [13]:
compared_data['mean crash'] = crash_data_describe_transpose['mean']
compared_data['mean norm'] = norm_data_describe_transpose['mean']
compared_data = compared_data.rename({'Mean: Crash - Norm': 'mean'})

In [14]:
compared_data_transpose = compared_data.transpose()

In [15]:
compared_data_transpose = compared_data_transpose.drop(columns={'Vehicles', 'Driver Age', 'Condition_Code', 'MorF'})

In [16]:
compared_data_transpose

,Unnamed: 0,Date,Direction,Time,TRAFFIC,PRCP,SNOW,SNWD,SNOW_DAY_SUM,DayOfWeek
mean,174.969464,1.583717e+02,0.000173,1.379718,323.532515,0.010749,0.169333,0.879970,0.377704,0.053846
mean crash,83812.367429,2.019120e+07,0.499948,12.805259,1370.384098,0.046074,0.551734,3.986624,3.170991,4.067295
mean norm,83637.397965,2.019104e+07,0.499775,11.425541,1046.851583,0.035325,0.382401,3.106654,2.793287,4.013449
